## Feature Engineering - Diabetes

### By:
Maria Camila Aristizábal Aguirre

### Date:
2026-08-18

## 📚 Import  libraries

In [1]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

## 💾 Load data

In [2]:
DATA_INTERMEDIA = Path("../../data/02_intermediate/diabetes_type_fixed.parquet")
df = pd.read_parquet(DATA_INTERMEDIA)
df.shape

(994, 9)

In [9]:
df.isna().sum()

Pregnancies                   6
Glucose                      10
BloodPressure                40
SkinThickness               242
Insulin                     387
BMI                          13
DiabetesPedigreeFunction      4
Age                           3
Outcome                       0
dtype: int64

## 👷 Data preparation or Feature Engineering

## Decisiones:

- El outlier extremo de `SkinThickness` (~99) identificado en el EDA se deja sin
  modificar: no hay evidencia de que sea un error de captura (a diferencia del bug
  de escala de `DiabetesPedigreeFunction`, confirmado y corregido en la exploración inicial), así
  que tratarlo como error sería un supuesto no justificado.
- No se aplica escalado (`StandardScaler`) en esta tarea; se decide en el paso siguiente que son los modelos,
  según el modelo baseline elegido.
- No se crean variables nuevas (no hay variables categóricas para combinar ni
  justificación clara para features derivadas en este dataset), por lo que no aplica en este proyecto.
- Orden de limpieza: primero se eliminan las filas sin `Outcome` (no sirven ni para
  entrenar ni para evaluar), luego los duplicados sobre lo que queda — así se evita
  eliminar por accidente la copia con `Outcome` conocido de una fila duplicada.
- El split train/test se hace **antes** de ajustar el imputador, y el imputador se
  ajusta **solo con train**, para evitar data leakage.

## Limpieza: filas sin Outcome y duplicados

In [3]:
n_inicial = len(df)

df = df[df["Outcome"].notna()].copy()
n_sin_target = n_inicial - len(df)

n_antes_dedup = len(df)
df = df.drop_duplicates().reset_index(drop=True)
n_duplicados = n_antes_dedup - len(df)

print(f"Filas eliminadas por Outcome faltante: {n_sin_target}")
print(f"Filas eliminadas por duplicados: {n_duplicados}")
print(f"Filas finales: {len(df)}")

Filas eliminadas por Outcome faltante: 19
Filas eliminadas por duplicados: 188
Filas finales: 787


In [4]:
DATA_PRIMARY = Path("../../data/03_primary/diabetes_clean.parquet")
DATA_PRIMARY.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(DATA_PRIMARY, index=False)

## Separación train/test (antes de imputar, para evitar data leakage)

In [5]:
COLUMNAS_FEATURES = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age",
]
TARGET = "Outcome"
PROPORCION_TEST = 0.2
SEMILLA = 42

X = df[COLUMNAS_FEATURES]
y = df[TARGET].astype(bool)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=PROPORCION_TEST, stratify=y, random_state=SEMILLA
)

print(f"Train: {X_train.shape[0]} filas | Test: {X_test.shape[0]} filas")
print("\nProporción de Outcome en train:")
print(y_train.value_counts(normalize=True))
print("\nProporción de Outcome en test:")
print(y_test.value_counts(normalize=True))

Train: 629 filas | Test: 158 filas

Proporción de Outcome en train:
Outcome
False    0.647059
True     0.352941
Name: proportion, dtype: float64

Proporción de Outcome en test:
Outcome
False    0.64557
True     0.35443
Name: proportion, dtype: float64


## Imputación (mediana), ajustada solo con train

In [10]:
preprocesador = ColumnTransformer(
    transformers=[
        ("imputer_mediana", SimpleImputer(strategy="median"), COLUMNAS_FEATURES),
    ],
    verbose_feature_names_out=False,
)
preprocesador.set_output(transform="pandas")

preprocesador.fit(X_train)

X_train_imputado = preprocesador.transform(X_train)[COLUMNAS_FEATURES]
X_test_imputado = preprocesador.transform(X_test)[COLUMNAS_FEATURES]

print("Nulos en X_train tras imputación:")
print(X_train_imputado.isna().sum())
print("\nNulos en X_test tras imputación:")
print(X_test_imputado.isna().sum())

Nulos en X_train tras imputación:
Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
dtype: int64

Nulos en X_test tras imputación:
Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
dtype: int64


In [11]:
train_final = X_train_imputado.assign(Outcome=y_train)
test_final = X_test_imputado.assign(Outcome=y_test)

MODEL_INPUT_DIR = Path("../../data/05_model_input")
MODEL_INPUT_DIR.mkdir(parents=True, exist_ok=True)
train_final.to_parquet(MODEL_INPUT_DIR / "train.parquet", index=False)
test_final.to_parquet(MODEL_INPUT_DIR / "test.parquet", index=False)

print(f"train.parquet: {train_final.shape}")
print(f"test.parquet: {test_final.shape}")

train.parquet: (629, 9)
test.parquet: (158, 9)


In [12]:
MODELS_DIR = Path("../../data/06_models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(preprocesador, MODELS_DIR / "preprocesador_imputacion.joblib")

['../../data/06_models/preprocesador_imputacion.joblib']

## 📊 Hallazgos - Feature Engineering


### Limpieza
- Se eliminaron 19 filas sin `Outcome` (no sirven para entrenar ni evaluar).
- Sobre las 975 filas restantes, se eliminaron 188 duplicados exactos (19.3%),
  quedando un dataset limpio de **787 filas**.

### Missingness (nulos reales, no ceros disfrazados)
| Columna | Nulos | % |
|---|---|---|
| Insulin | 387 | 49.2% |
| SkinThickness | 242 | 30.8% |
| BloodPressure | 40 | 5.1% |
| BMI | 13 | 1.7% |
| Glucose | 10 | 1.3% |
| Pregnancies | 6 | 0.8% |
| DiabetesPedigreeFunction | 4 | 0.5% |
| Age | 3 | 0.4% |

Insulin y SkinThickness confirman el nivel de missingness ya detectado en el EDA.
Pregnancies, DiabetesPedigreeFunction y Age son un hallazgo nuevo: nulos
genuinos no capturados en las validaciones del entendimiento del problema y la exploración inicial de datos (que solo revisaban ceros
disfrazados y el bug de escala de DPF). Por eso el imputador se aplicó finalmente a las
8 variables numéricas (no solo a las 5 detectadas originalmente), evitando dejar nulos
sin tratar por una lista incompleta.

### Split train/test
- 629 filas de train (64.7% False / 35.3% True) y 158 de test (64.6% False / 35.4% True),
  proporciones prácticamente idénticas gracias a `stratify=Outcome`.

### Decisiones registradas
- El outlier extremo de SkinThickness (~99) se deja sin modificar: no hay evidencia de
  que sea un error de captura.
- No se aplica escalado (`StandardScaler`) en esta tarea; queda pendiente para la Tarea 5
  según el modelo baseline elegido.
- No se generan variables nuevas; `data/04_feature/` no se usa en este proyecto.